In [ ]:
import kagglehub

spotify_path = kagglehub.dataset_download("yamaerenay/spotify-dataset-19212020-600k-tracks")
billboard_path = kagglehub.dataset_download("elizabethearhart/billboard-hot-1001958-2024")

print("Path to Spotify dataset files:", spotify_path)
print("Path to Billboard dataset files:", billboard_path)

In [ ]:
import pandas as pd
import ast
import re

tracks = pd.read_csv(f"{spotify_path}/tracks.csv")
billboard_tracks = pd.read_csv(f"{billboard_path}/hot-100-current.csv")

def normalize_key(s):
    return (
        s.str.normalize('NFKD')
         .str.encode('ascii', errors='ignore').str.decode('ascii')
         .str.lower()
         .str.strip()
    )

COLLAB_SEP = re.compile(r'\s+(?:feat\.?|featuring|with|&|,|/)\s+', flags=re.IGNORECASE)
PAREN = re.compile(r'\s*[\(\[].*?[\)\]]\s*')
SUFFIX = re.compile(r'\s+-\s+.+$')

def primary_artist(s):
    if pd.isna(s):
        return s
    return COLLAB_SEP.split(s, maxsplit=1)[0].strip()

def clean_title(s):
    return s.str.replace(PAREN, '', regex=True).str.replace(SUFFIX, '', regex=True)

tracks['artists'] = tracks['artists'].apply(ast.literal_eval)
tracks['name_clean'] = clean_title(tracks['name'])
tracks['track_id'] = (tracks['name_clean'] + '|' + tracks['artists'].str[0]).transform(normalize_key)
tracks['artists'] = tracks['artists'].apply(frozenset)
tracks = tracks.sort_values('popularity', ascending=False).drop_duplicates(subset=['track_id'], keep='first').reset_index(drop=True)
billboard_tracks['primary_artist'] = billboard_tracks['performer'].apply(primary_artist)
billboard_tracks['title_clean'] = clean_title(billboard_tracks['title'])
billboard_tracks['track_id'] = (billboard_tracks['title_clean'] + '|' + billboard_tracks['primary_artist']).transform(normalize_key)
billboard_tracks = billboard_tracks.sort_values('peak_pos', ascending=True).drop_duplicates(subset=['track_id'], keep='first').reset_index(drop=True)
df = tracks.merge(billboard_tracks[['track_id', 'peak_pos', 'wks_on_chart']], on="track_id", how="left", validate="many_to_one")
df['is_hit'] = df['peak_pos'].notnull()

def hit_tier(peak):
    if pd.isna(peak): return 0
    if peak == 1: return 4
    if peak <= 10: return 3
    if peak <= 50: return 2
    return 1

df['hit_tier'] = df['peak_pos'].apply(hit_tier)
df['release_year'] = pd.to_datetime(df['release_date'], format='mixed').dt.year
df['release_decade'] = (df['release_year'] // 10) * 10
df = df[df['release_year'] >= 1958].reset_index(drop=True)
charted = df[df['is_hit']]


In [ ]:
print('--SHAPE--')
print(df.shape)
print('--HEAD--')
print(df.head())
print('--DATA TYPES--')
print(df.dtypes)
print('--NULL COUNT--')
print(df.isnull().sum())
print('--DUPLICATE COUNT--')
print(df.duplicated(subset=['track_id']).sum())
print('--DESCRIPTION--')
print(df.describe())

In [ ]:
print(df['is_hit'].value_counts())
print(f"\nHit rate: {df['is_hit'].mean():.2%} — heavily imbalanced, so we'll use PR-AUC and tune the decision threshold.")

In [ ]:
import altair as alt
import numpy as np
from pathlib import Path

# Cells below pre-aggregate large data where possible, but a couple of
# the scatters keep a few thousand sampled rows inline. Default cap is 5000.
alt.data_transformers.disable_max_rows()

features = ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'valence', 'tempo']

numeric_cols = df[features].columns.tolist()
hist_rows = []
for col in numeric_cols:
    vals = df[col].dropna()
    counts, edges = np.histogram(vals, bins=50)
    centers = (edges[:-1] + edges[1:]) / 2
    hist_rows.extend({'feature': col, 'value': float(c), 'count': int(ct)}
                     for c, ct in zip(centers, counts))
hist_df = pd.DataFrame(hist_rows)

chart = (alt.Chart(hist_df)
    .mark_bar()
    .encode(
        x=alt.X('value:Q', axis=alt.Axis(labels=False, ticks=False, title=None)),
        y=alt.Y('count:Q', axis=alt.Axis(labels=False, ticks=False, title=None)),
    )
    .properties(width=140, height=80)
    .facet(facet=alt.Facet('feature:N', title=None), columns=4)
    .resolve_scale(x='independent', y='independent')
)

out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/feature-distribution.json'
out.write_text(chart.to_json(indent=None))
chart

In [ ]:
from scipy.stats import gaussian_kde

# Compute KDEs in pandas so the spec carries only the evaluated curves
# (200 points * 6 features * 4 tiers ~ 4800 rows) instead of 391k raw rows.
tier_specs = [
    ('Top 10',  df['hit_tier'] >= 3),
    ('Top 50',  df['hit_tier'] == 2),
    ('Top 100', df['hit_tier'] == 1),
    ('Not hit', df['hit_tier'] == 0),
]

density_rows = []
for feat in features:
    x = np.linspace(df[feat].min(), df[feat].max(), 200)
    for tier_name, mask in tier_specs:
        vals = df.loc[mask, feat].dropna().values
        if len(vals) < 2:
            continue
        y = gaussian_kde(vals)(x)
        density_rows.extend({'feature': feat, 'tier': tier_name,
                             'value': float(xi), 'density': float(yi)}
                            for xi, yi in zip(x, y))
density_df = pd.DataFrame(density_rows)

chart = (alt.Chart(density_df)
    .mark_area(opacity=0.25, line={'strokeWidth': 1.5})
    .encode(
        x=alt.X('value:Q', title=None),
        y=alt.Y('density:Q', title=None, stack=None, axis=alt.Axis(labels=False, ticks=False)),
        color=alt.Color('tier:N', title="Tier", sort=['Top 10', 'Top 50', 'Top 100', 'Not hit']),
    )
    .properties(width=180, height=120)
    .facet(facet=alt.Facet('feature:N', title=None), columns=3)
    .resolve_scale(x='independent', y='independent')
)
out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/feature-top-chart-relationship.json'
out.write_text(chart.to_json(indent=None))
chart

In [ ]:
# Faceted scatter: each audio feature vs weeks on chart, for charted songs.
# Sample down from ~14k to keep the spec under ~1MB.
charted_sample = charted[['wks_on_chart'] + features].sample(
    n=min(6000, len(charted)), random_state=42
)

(alt.Chart(charted_sample)
    .transform_fold(features, as_=['feature', 'value'])
    .mark_circle(size=15, opacity=0.2)
    .encode(
        x=alt.X('value:Q', title=None),
        y=alt.Y('wks_on_chart:Q', title='Weeks on Chart'),
    )
    .properties(width=180, height=140)
    .facet(facet=alt.Facet('feature:N', title=None), columns=3)
    .resolve_scale(x='independent')
)

In [ ]:
charted_sample = charted[['danceability', 'peak_pos', 'release_decade']].sample(
    n=min(8000, len(charted)), random_state=42
)

(alt.Chart(charted_sample)
    .mark_circle(opacity=0.35, size=25)
    .encode(
        x=alt.X('danceability:Q', title='Danceability'),
        y=alt.Y('peak_pos:Q', title='Peak Position'),
        color=alt.Color('release_decade:Q', title='Release Decade'),
    )
    .properties(width=560, height=320)
)

In [ ]:
hit_rate = (df.groupby('release_year')['is_hit'].mean()
              .reset_index()
              .rename(columns={'is_hit': 'hit_rate'}))
hit_rate['release_year'] = pd.to_datetime(hit_rate['release_year'], format='%Y')

chart = (alt.Chart(hit_rate)
    .mark_line()
    .encode(
        x=alt.X('release_year:T', title='Release Year', axis=alt.Axis(format='%Y')),
        y=alt.Y('hit_rate:Q', title='Hit Rate'),
    )
    .properties(width="container", height=240)
)
out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/hit-rate-over-time.json'
out.write_text(chart.to_json(indent=None))
chart

In [ ]:
corr_features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'valence', 'tempo']
corr = df[corr_features].corr().stack().reset_index()
corr.columns = ['x', 'y', 'value']

heatmap = alt.Chart(corr).mark_rect().encode(
    x=alt.X('x:N', title=None, sort=corr_features),
    y=alt.Y('y:N', title=None, sort=corr_features),
    color=alt.Color('value:Q', scale=alt.Scale(domain=[-1, 1], domainMid=0), title='Correlation'),
)
labels = alt.Chart(corr).mark_text(fontSize=10).encode(
    x=alt.X('x:N', sort=corr_features),
    y=alt.Y('y:N', sort=corr_features),
    text=alt.Text('value:Q', format='.2f'),
    color=alt.condition('abs(datum.value) > 0.5',
                        alt.value('white'),
                        alt.value('black')),
)
chart = (heatmap + labels).properties(width=400, height=400)
out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/feature-correlation.json'
out.write_text(chart.to_json(indent=None))
chart

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[features], df['is_hit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5, stratify=y)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Standardize all features (continuous and binary) so the L1 penalty
# treats them on the same scale and the coefficients are comparable.
logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(l1_ratio=1, solver='liblinear', max_iter=1000, random_state=5),
)

logreg_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [None, 5, 7, 10, 20],
    'min_samples_split': [2, 5, 10],
}

decision_tree_model = DecisionTreeClassifier(random_state=5)
grid = GridSearchCV(decision_tree_model, param_grid, scoring='average_precision', cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best PR-AUC: {grid.best_score_:.4f}")
decision_tree_model = grid.best_estimator_

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'max_depth': [None, 5, 7, 10, 20],
    'min_samples_split': [2, 5, 10],
}

random_forest_model = RandomForestClassifier(random_state=5)
grid = GridSearchCV(random_forest_model, param_grid, scoring='average_precision', cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best PR-AUC: {grid.best_score_:.4f}")
random_forest_model = grid.best_estimator_

In [ ]:
import xgboost as xgb

param_grid = {
    'max_depth': [5, 7, 10],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [200, 400],
}

xgb_model = xgb.XGBClassifier(tree_method='hist', random_state=5, eval_metric='aucpr')
grid = GridSearchCV(xgb_model, param_grid, scoring='average_precision', cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best PR-AUC: {grid.best_score_:.4f}")
xgb_model = grid.best_estimator_

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

for name, model in [('logreg', logreg_pipeline), ('dt', decision_tree_model), ('rf', random_forest_model), ('xgb', xgb_model)]:
    proba = model.predict_proba(X_test)[:, 1]
    print(f"{name} test PR-AUC: {average_precision_score(y_test, proba):.4f}")

In [ ]:
models = {'logreg': logreg_pipeline, 'dt': decision_tree_model, 'rf': random_forest_model, 'xgb': xgb_model}
test_scores = {name: average_precision_score(y_test, m.predict_proba(X_test)[:, 1]) for name, m in models.items()}
best_name = max(test_scores, key=test_scores.get)
best_model = models[best_name]
print(f"Picking {best_name} (test PR-AUC = {test_scores[best_name]:.4f}) for threshold tuning\n")

proba = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, proba)

precision_floor = 0.25
mask = precision[:-1] >= precision_floor
if mask.any():
    j = np.where(mask)[0][recall[:-1][mask].argmax()]
    chosen_t = thresholds[j]
    print(f"Operating point: threshold={chosen_t:.4f}")
    print(f"  precision={precision[j]:.3f}  recall={recall[j]:.3f}  flagged={(proba >= chosen_t).sum()}")
else:
    print(f"No threshold achieves precision >= {precision_floor}; try lowering the floor.")

pr_df = pd.DataFrame({'recall': recall[:-1], 'precision': precision[:-1]})
base = float(y_test.mean())

curve = alt.Chart(pr_df).mark_line().encode(
    x=alt.X('recall:Q', title='Recall'),
    y=alt.Y('precision:Q', title='Precision'),
)
base_rate = alt.Chart(pd.DataFrame({'y': [base]})).mark_rule(strokeDash=[4, 3]).encode(y='y:Q')

chart = alt.layer(curve, base_rate).properties(
    width="container", height=320,
    title=f'PR Curve: {best_name} on Test Set'
)

out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/pr-curve.json'
out.write_text(chart.to_json(indent=None))
chart

In [ ]:
# LR coefficients (direction + magnitude, signed). All features are standardized,
# so coefficients are comparable across both continuous and binary features.
lr = logreg_pipeline.named_steps['logisticregression']
lr_features = list(X_train.columns)
lr_imp = pd.DataFrame({'feature': lr_features, 'coefficient': lr.coef_[0]})
lr_imp = lr_imp.reindex(lr_imp['coefficient'].abs().sort_values(ascending=False).index).reset_index(drop=True)
lr_imp['sign'] = lr_imp['coefficient'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

# Tree-based importances. sklearn's feature_importances_ returns mean decrease
# in impurity (sample-weighted); XGBoost's returns mean gain per split. We
# surface the metric on each subplot's x-axis so the bars aren't read as the
# same quantity. concat (rather than facet) gives us per-chart axis titles.
tree_models = [
    ('Decision Tree', decision_tree_model, 'Mean decrease in impurity'),
    ('Random Forest', random_forest_model, 'Mean decrease in impurity'),
    ('XGBoost', xgb_model, 'Mean gain per split'),
]
tree_imp = pd.concat([
    pd.DataFrame({
        'feature': m.feature_names_in_,
        'importance': m.feature_importances_,
        'model': name,
    })
    for name, m, _ in tree_models
], ignore_index=True)

print('--- LR coefficients (sorted by |coefficient|) ---')
print(lr_imp.to_string(index=False))
print('\n--- Tree-based importance ---')
print(tree_imp.pivot(index='feature', columns='model', values='importance').to_string())

lr_chart = (alt.Chart(lr_imp)
    .mark_bar()
    .encode(
        x=alt.X('coefficient:Q', title='Coefficient'),
        y=alt.Y('feature:N', title=None, sort=list(lr_imp['feature'])),
        color=alt.Color('sign:N', title=None, sort=['Positive', 'Negative']),
    )
    .properties(width="container", height=300, title='LR (signed)')
)

def tree_subchart(name, model, metric):
    subset = tree_imp[tree_imp['model'] == name]
    return (alt.Chart(subset)
        .mark_bar()
        .encode(
            x=alt.X('importance:Q', title=metric),
            y=alt.Y('feature:N', title=None, sort='-x'),
        )
        .properties(width=180, height=300, title=name)
    )

tree_chart = alt.concat(
    *[tree_subchart(name, m, metric) for name, m, metric in tree_models],
    columns=3,
)

out_dir = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs'
(out_dir / 'feature-importance-lr.json').write_text(lr_chart.to_json(indent=None))
(out_dir / 'feature-importance-trees.json').write_text(tree_chart.to_json(indent=None))

lr_chart & tree_chart

In [ ]:
# 1D scatter of energy stratified by hit/not-hit, faceted as two parallel strips.
# Downsample non-hits to match the hit count so the visual isn't drowned by the 96% non-hit majority.
# Jitter on y just spreads points vertically within each strip to avoid overplotting.
hits = df[df['is_hit']]
not_hits = df[~df['is_hit']].sample(n=len(hits), random_state=42)
balanced = pd.concat([hits, not_hits], ignore_index=True)
balanced['jitter'] = np.random.default_rng(42).uniform(0, 1, len(balanced))
balanced['class'] = balanced['is_hit'].map({True: 'Hit', False: 'Not hit'})

chart = (alt.Chart(balanced[['energy', 'jitter', 'class']])
    .mark_circle(opacity=0.25, size=10)
    .encode(
        x=alt.X('energy:Q', title='Energy'),
        y=alt.Y('jitter:Q', title=None, axis=None),
        color=alt.Color('class:N', title=None, sort=['Hit', 'Not hit']),
    )
    .properties(width=480, height=60)
    .facet(row=alt.Row('class:N', title=None, header=alt.Header(labelAngle=0, labelAlign='left', labelPadding=15)))
)
out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/energy-hit.json'
out.write_text(chart.to_json(indent=None))
chart

In [ ]:
# 1D scatter of instrumentalness stratified by hit/not-hit, faceted as two parallel strips.
# Same approach as the energy plot above.
hits = df[df['is_hit']]
not_hits = df[~df['is_hit']].sample(n=len(hits), random_state=42)
balanced = pd.concat([hits, not_hits], ignore_index=True)
balanced['jitter'] = np.random.default_rng(42).uniform(0, 1, len(balanced))
balanced['class'] = balanced['is_hit'].map({True: 'Hit', False: 'Not hit'})

chart = (alt.Chart(balanced[['instrumentalness', 'jitter', 'class']])
    .mark_circle(opacity=0.25, size=10)
    .encode(
        x=alt.X('instrumentalness:Q', title='Instrumentalness'),
        y=alt.Y('jitter:Q', title=None, axis=None),
        color=alt.Color('class:N', title=None, sort=['Hit', 'Not hit']),
    )
    .properties(width=480, height=60)
    .facet(row=alt.Row('class:N', title=None, header=alt.Header(labelAngle=0, labelAlign='left', labelPadding=15)))
)
out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/hit-songs/instrumentalness-hit.json'
out.write_text(chart.to_json(indent=None))
chart